# Task 3: MuCoCo reproduction (Part 1) and MuReCo prototype (Part 2) · v2
Akwasi Asare · PhD assessment, Prof. Ezekiel Soremekun (SUTD)

**Runtime:** `Runtime > Change runtime type > A100 GPU`. Put the **new** `mureco.zip` in `MyDrive/` (replace the old one).

What changed in v2: a more tolerant answer parser (markdown, code fences, multi-line lists, arrows, trailing explanations),
parse diagnostics in every report, non-literal variables (iterators, `map` objects) are no longer traced, and A100 settings:
the paper's own Qwen2.5-Coder-14B for Part 1, and the 1.5B and 7B models from the slides for Part 2.

| Step | What it does | Time on an A100 (approx.) |
|---|---|---|
| 0 | Setup | 3 min |
| 1 | Parser tests and pipeline self-test (no model) | 3 min |
| 2 | **Part 1**: MuCoCo's experiment with Qwen2.5-Coder-14B (the paper's model) | 10 min |
| 3 | **Part 2**: MuReCo on Qwen2.5-Coder-1.5B and 7B | 20–30 min |
| 4 | Diagnostics: parse rates and raw model answers | 1 min |
| 5 | Part 2b (optional): reasoning model | 30 min |
| 6 | Live walkthrough, save results | 3 min |

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 0. Setup (fresh copy of the code from Drive)

In [ ]:
%cd /content
!rm -rf /content/mureco
![ -d /content/mucoco ] || git clone -q --depth 1 https://github.com/mucoco-tester/mucoco-tester.github.io.git /content/mucoco
from google.colab import drive
drive.mount('/content/drive')
!unzip -q -o /content/drive/MyDrive/mureco.zip -d /content && ls /content/mureco
%env MUCOCO_DIR=/content/mucoco
%cd /content/mureco
!pip -q install -U "transformers>=4.44" accelerate bitsandbytes

## 1. Tests (no LLM)
Parser tests on messy formats, then the pipeline self-test with test doubles.

In [ ]:
!python scripts/parser_test.py | tail -1
!python scripts/selftest.py 2>&1 | grep -v DEBUG | tail -4

## 2. Part 1: MuCoCo's experiment with MuCoCo's own code, on the paper's model
CruxEval output prediction, all 11 MuCoCo operators, MuCoCo's zero-shot prompt, answer processing and correctness check,
with **Qwen2.5-Coder-14B-Instruct**, the model used in the paper (Table 4).

In [ ]:
!python scripts/run_mucoco_demo.py --model Qwen/Qwen2.5-Coder-14B-Instruct --n 50 --batch 32 --out results_part1 2>&1 | grep -v DEBUG

Optional, artifact-level reproduction of Table 6 (no GPU): download the replication package from
[Figshare](https://figshare.com/articles/journal_contribution/_b_MuCoCo_Code_Consistency_Testing_Framework_Replication_Package_b_/30402541),
place it at `mucoco/MuCoCo_results/MuCoCo_experiment_results`, and run `mucoco/MuCoCo_results/notebooks/RQ1_results_aggregation.ipynb`.

## 3. Part 2: MuReCo prototype
Originals + single MuCoCo mutations + all valid pairs; the model outputs a trace and the output; traces are checked
against real execution; each original is rerun twice (different batch) as a noise baseline.

In [ ]:
!python scripts/run_mureco.py --models Qwen/Qwen2.5-Coder-1.5B-Instruct Qwen/Qwen2.5-Coder-7B-Instruct --n 60 --batch 32 --max-new-tokens 768 --out results_part2 2>&1 | grep -v DEBUG

In [ ]:
from IPython.display import Markdown, Image, display
display(Image('results_part2/inconsistency_rates.png'))
display(Markdown(open('results_part2/REPORT.md').read()))

## 4. Diagnostics: are low scores the model or the parser?
Shows parse rates and three raw answers per model, with how they were scored.

In [ ]:
import json, glob, textwrap
for f in sorted(glob.glob('results_part2*/*_summary.json')):
    s = json.load(open(f)); p = s['parsing']
    print(f"{s['model']}: no readable output {p['invalid_output_rate'][0]:.1f}%, all traces readable {p['trace_parsed_rate'][0]:.1f}%")
for f in sorted(glob.glob('results_part2*/*_records.jsonl')):
    print('\n' + '=' * 100 + '\n' + f)
    for line in open(f).readlines()[:3]:
        r = json.loads(line); o = r['variants']['original']
        print(f"\n--- {r['task']}  tracked {r['tracked']}  expected {r['expected']}")
        print('ground truth:', r['gt_trace'])
        print('RAW MODEL ANSWER:\n' + textwrap.indent(o['raw'][-900:], '    '))
        print(f"scored -> output={o['output']} ({o['outcome']}), traces={o['traces']}, trace correct={o['trace_correct']}")

## 5. Part 2b (optional): a reasoning model
DeepSeek-R1-Distill-Qwen-1.5B thinks before answering; it needs a large token budget (answers cut off mid-thought count as invalid).

In [ ]:
!python scripts/run_mureco.py --models deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B --n 20 --max-new-tokens 4096 --batch 24 --reruns 1 --out results_part2_reasoning 2>&1 | grep -v DEBUG
from IPython.display import Markdown, display
display(Markdown(open('results_part2_reasoning/REPORT.md').read()))

## 6. Live walkthrough for the demo, then save everything

In [ ]:
!python scripts/walkthrough.py --task CruxEval_0 --variant random+for2while --model Qwen/Qwen2.5-Coder-7B-Instruct 2>&1 | grep -v DEBUG

In [ ]:
!zip -q -r /content/task3_results.zip results_part1 results_part2 results_part2_reasoning 2>/dev/null; ls -la /content/task3_results.zip
!cp /content/task3_results.zip /content/drive/MyDrive/task3_results.zip && echo "Saved to /content/drive/MyDrive/task3_results.zip"
from google.colab import files; files.download('/content/task3_results.zip')